## 췌장암 병리 생존예측 데이터 검증

### 목적
본 노트북은 췌장암 병리 기반 생존예측 모델 개발 전, TCGA-PAAD 및 CPTAC-PDAC 데이터의 파일 구성, 환자 단위 매칭, 생존 라벨 품질을 점검한다.

### 입력 데이터
- 기본 경로: `../../data/pancreatic_cancer_pathology`
- TCGA-PAAD: WSI SVS, RNA-seq STAR counts, matched clinical/survival tables
- CPTAC-PDAC: WSI DICOM series, omics, matched clinical table

### 주요 확인 항목
- 파일 인벤토리 및 확장자별 개수
- 메타데이터 테이블 shape, column, 결측치
- 생존 시간 및 censoring/event 분포
- 환자 ID 기준 중복 및 WSI/RNA/clinical 매칭 상태
- 생존예측 모델용 초기 cohort candidate 저장

### 출력
- `../../results/pancreatic_cancer_pathology/data_verification/file_inventory.csv`
- `../../results/pancreatic_cancer_pathology/data_verification/extension_summary.csv`
- `../../results/pancreatic_cancer_pathology/data_verification/table_summary.csv`
- `../../results/pancreatic_cancer_pathology/data_verification/tcga_survival_cohort_candidate.csv`
- `../../results/pancreatic_cancer_pathology/data_verification/cptac_survival_cohort_candidate.csv`

### 논문 작성 메모
본 단계에서는 원본 데이터를 수정하지 않고, 공개 코호트별 병리 WSI와 생존 라벨의 환자 단위 매칭 가능성을 평가한다.

## 1. 환경 설정

In [ ]:
from pathlib import Path
from typing import Optional
import random

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_PATH = Path("../../data")
PROJECT_DATA_PATH = DATA_PATH / "pancreatic_cancer_pathology"
RAW_PATH = PROJECT_DATA_PATH / "raw"
TCGA_PATH = RAW_PATH / "TCGA_PAAD"
CPTAC_PATH = RAW_PATH / "CPTAC_PDAC"
RESULT_PATH = Path("../../results")
OUTPUT_DIR = RESULT_PATH / "pancreatic_cancer_pathology" / "data_verification"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH: {DATA_PATH.resolve()}")
print(f"PROJECT_DATA_PATH exists: {PROJECT_DATA_PATH.exists()} | {PROJECT_DATA_PATH}")
print(f"TCGA_PATH exists: {TCGA_PATH.exists()} | {TCGA_PATH}")
print(f"CPTAC_PATH exists: {CPTAC_PATH.exists()} | {CPTAC_PATH}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")

## 2. 공통 함수

In [ ]:
def get_file_inventory(base_path: Path) -> pd.DataFrame:
    records = []
    for path in sorted(base_path.rglob("*")):
        if not path.is_file():
            continue
        rel_path = path.relative_to(base_path)
        records.append(
            {
                "relative_path": rel_path.as_posix(),
                "parent": rel_path.parent.as_posix(),
                "file_name": path.name,
                "extension": path.suffix.lower().lstrip(".") or "[no_ext]",
                "size_mb": path.stat().st_size / (1024 ** 2),
            }
        )
    return pd.DataFrame(records)


def read_table(path: Path, nrows=None) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".tsv" or suffix == ".cct" or suffix == ".txt":
        return pd.read_csv(path, sep="\t", nrows=nrows)
    if suffix == ".csv":
        return pd.read_csv(path, nrows=nrows)
    raise ValueError(f"지원하지 않는 테이블 확장자입니다: {path}")


def summarize_table(path: Path) -> dict:
    df = read_table(path)
    survival_keywords = ("survival", "vital", "death", "follow", "os", "days")
    id_keywords = ("patient", "case", "sample", "file", "series")
    survival_cols = [c for c in df.columns if any(k in c.lower() for k in survival_keywords)]
    id_cols = [c for c in df.columns if any(k in c.lower() for k in id_keywords)]
    return {
        "path": path.as_posix(),
        "n_rows": len(df),
        "n_cols": df.shape[1],
        "n_duplicated_rows": int(df.duplicated().sum()),
        "columns": " | ".join(map(str, df.columns)),
        "id_candidate_columns": " | ".join(id_cols),
        "survival_candidate_columns": " | ".join(survival_cols),
    }


def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(
        {
            "missing_count": df.isna().sum(),
            "missing_rate": df.isna().mean(),
            "dtype": df.dtypes.astype(str),
        }
    )
    return out.sort_values(["missing_count", "missing_rate"], ascending=False)


def find_existing_file(root: Path, file_name: str) -> Optional[Path]:
    matches = list(root.rglob(file_name))
    if len(matches) == 0:
        return None
    if len(matches) > 1:
        print(f"주의: 동일 파일명 후보가 여러 개입니다. 첫 번째를 사용합니다: {file_name}")
    return matches[0]

## 3. 파일 인벤토리 확인

### 목적
전체 데이터 파일 수, 확장자, 용량 분포를 확인하여 병리 WSI와 메타데이터가 기대한 위치에 있는지 점검한다.

In [ ]:
file_inventory = get_file_inventory(PROJECT_DATA_PATH)
file_inventory.to_csv(OUTPUT_DIR / "file_inventory.csv", index=False)

extension_summary = (
    file_inventory.groupby("extension", as_index=False)
    .agg(n_files=("relative_path", "count"), total_size_mb=("size_mb", "sum"), median_size_mb=("size_mb", "median"))
    .sort_values("n_files", ascending=False)
)
extension_summary.to_csv(OUTPUT_DIR / "extension_summary.csv", index=False)

print(file_inventory.shape)
display(extension_summary)
display(file_inventory.head(10))

In [ ]:
directory_summary = (
    file_inventory.assign(top_dir=file_inventory["relative_path"].str.split("/").str[:3].str.join("/"))
    .groupby(["top_dir", "extension"], as_index=False)
    .agg(n_files=("relative_path", "count"), total_size_mb=("size_mb", "sum"))
    .sort_values(["top_dir", "n_files"], ascending=[True, False])
)
directory_summary.to_csv(OUTPUT_DIR / "directory_summary.csv", index=False)
display(directory_summary.head(50))

## 4. 메타데이터 테이블 요약

### 목적
CSV/TSV/CCT/TXT 중 분석 메타데이터 후보를 읽어 shape, ID 후보 컬럼, 생존 후보 컬럼을 확인한다. RNA-seq 개별 count 파일은 수가 많으므로 이 단계의 상세 요약에서는 제외한다.

In [ ]:
table_paths = []
for suffix in ("*.csv", "*.tsv", "*.cct", "*.txt"):
    table_paths.extend(PROJECT_DATA_PATH.rglob(suffix))

metadata_table_paths = [
    p for p in sorted(table_paths)
    if "RNA_SEQ_STAR_COUNTS" not in p.as_posix() and p.name != "annotations.txt"
]

table_summary = pd.DataFrame([summarize_table(p) for p in metadata_table_paths])
table_summary.to_csv(OUTPUT_DIR / "table_summary.csv", index=False)

print(f"metadata table count: {len(metadata_table_paths)}")
display(table_summary[["path", "n_rows", "n_cols", "id_candidate_columns", "survival_candidate_columns"]])

## 5. TCGA-PAAD 생존 라벨 및 WSI 매칭 확인

### 목적
TCGA-PAAD에서 환자 단위로 diagnostic WSI 1개, RNA-seq, 임상 생존 라벨이 모두 매칭되는 초기 cohort candidate를 구성한다.

In [ ]:
tcga_matched_path = TCGA_PATH / "TCGA_PAAD_matched"
tcga_patient_path = tcga_matched_path / "tcga_paad_matched_patient_table.csv"
tcga_one_wsi_path = tcga_matched_path / "tcga_paad_matched_patient_table_dx_one_per_patient.csv"
tcga_wsi_all_path = tcga_matched_path / "tcga_paad_matched_wsi_dx_all_files.csv"
tcga_rnaseq_path = tcga_matched_path / "tcga_paad_matched_rnaseq_files.csv"

tcga_patient_df = pd.read_csv(tcga_patient_path)
tcga_one_wsi_df = pd.read_csv(tcga_one_wsi_path)
tcga_wsi_all_df = pd.read_csv(tcga_wsi_all_path)
tcga_rnaseq_df = pd.read_csv(tcga_rnaseq_path)

print("patient table:", tcga_patient_df.shape)
print("one WSI per patient:", tcga_one_wsi_df.shape)
print("all diagnostic WSI:", tcga_wsi_all_df.shape)
print("RNA-seq files:", tcga_rnaseq_df.shape)
display(tcga_one_wsi_df.head())

In [ ]:
tcga_survival_cols = [
    "patient_id", "vital_status", "days_to_death", "days_to_last_follow_up",
    "OS_time", "OS_event", "age_at_diagnosis", "primary_diagnosis",
    "tumor_grade", "ajcc_pathologic_stage", "ajcc_pathologic_t",
    "ajcc_pathologic_n", "ajcc_pathologic_m",
]

print("patient_id duplicated:", tcga_one_wsi_df["patient_id"].duplicated().sum())
print("OS_time missing:", tcga_one_wsi_df["OS_time"].isna().sum())
print("OS_event missing:", tcga_one_wsi_df["OS_event"].isna().sum())
print("OS_event distribution")
display(tcga_one_wsi_df["OS_event"].value_counts(dropna=False).rename_axis("OS_event").reset_index(name="n"))

display(tcga_one_wsi_df[tcga_survival_cols].describe(include="all"))
display(missing_summary(tcga_one_wsi_df[tcga_survival_cols]).head(20))

In [ ]:
tcga_cohort = tcga_one_wsi_df.copy()
tcga_cohort["wsi_path"] = tcga_cohort["selected_wsi_file_name"].apply(
    lambda x: find_existing_file(TCGA_PATH / "WSI_SVS", x)
)
tcga_cohort["wsi_exists"] = tcga_cohort["wsi_path"].notna()
tcga_cohort["wsi_path"] = tcga_cohort["wsi_path"].astype(str)

tcga_model_cols = [
    "patient_id", "selected_wsi_sample_id", "selected_wsi_file_id",
    "selected_wsi_file_name", "wsi_path", "wsi_exists", "n_dx_wsi_files",
    "n_rnaseq_files", "OS_time", "OS_event", "vital_status",
    "days_to_death", "days_to_last_follow_up", "age_at_diagnosis",
    "gender", "race", "ethnicity", "primary_diagnosis", "tumor_grade",
    "ajcc_pathologic_stage", "ajcc_pathologic_t", "ajcc_pathologic_n", "ajcc_pathologic_m",
]
tcga_cohort = tcga_cohort[tcga_model_cols]
tcga_cohort.to_csv(OUTPUT_DIR / "tcga_survival_cohort_candidate.csv", index=False)

print("TCGA cohort candidate:", tcga_cohort.shape)
print("WSI exists distribution")
display(tcga_cohort["wsi_exists"].value_counts(dropna=False).rename_axis("wsi_exists").reset_index(name="n"))
display(tcga_cohort.head())

## 6. CPTAC-PDAC 생존 라벨 및 WSI series 매칭 확인

### 목적
CPTAC-PDAC에서 환자 단위 clinical survival label과 WSI DICOM series 수를 확인한다. CPTAC는 case 단위로 여러 series가 있으므로, 모델 개발 전 tumor/normal series 구분 기준을 명확히 해야 한다.

In [ ]:
cptac_matched_path = CPTAC_PATH / "matched"
cptac_clinical_path = cptac_matched_path / "cptac_pda_matched_clinical.csv"
cptac_summary_path = cptac_matched_path / "cptac_pda_matched_case_summary.csv"
cptac_wsi_path = cptac_matched_path / "cptac_pda_matched_wsi_series.csv"

cptac_clinical_df = pd.read_csv(cptac_clinical_path)
cptac_summary_df = pd.read_csv(cptac_summary_path)
cptac_wsi_df = pd.read_csv(cptac_wsi_path)

print("clinical:", cptac_clinical_df.shape)
print("case summary:", cptac_summary_df.shape)
print("WSI series:", cptac_wsi_df.shape)
display(cptac_summary_df.head())
display(cptac_wsi_df.head())

In [ ]:
cptac_survival = cptac_summary_df.copy()
cptac_survival["OS_time"] = pd.to_numeric(cptac_survival["follow_up_days"], errors="coerce")
cptac_survival["OS_event"] = cptac_survival["vital_status"].str.lower().eq("deceased").astype(int)

wsi_series_summary = (
    cptac_wsi_df.groupby("case_id", as_index=False)
    .agg(
        n_wsi_series_observed=("SeriesInstanceUID", "nunique"),
        total_series_size_mb=("series_size_MB", "sum"),
        series_descriptions=("SeriesDescription", lambda x: " | ".join(sorted(set(map(str, x.dropna()))))),
    )
)
cptac_cohort = cptac_survival.merge(wsi_series_summary, on="case_id", how="left")
cptac_cohort["has_wsi_series"] = cptac_cohort["n_wsi_series_observed"].fillna(0).gt(0)

print("case_id duplicated:", cptac_cohort["case_id"].duplicated().sum())
print("OS_time missing:", cptac_cohort["OS_time"].isna().sum())
print("OS_event distribution")
display(cptac_cohort["OS_event"].value_counts(dropna=False).rename_axis("OS_event").reset_index(name="n"))
display(cptac_cohort["has_wsi_series"].value_counts(dropna=False).rename_axis("has_wsi_series").reset_index(name="n"))
display(missing_summary(cptac_cohort[["case_id", "OS_time", "OS_event", "follow_up_days", "vital_status", "histology_diagnosis", "age", "sex", "tumor_stage_pathological", "n_wsi_series_observed"]]).head(20))

In [ ]:
cptac_model_cols = [
    "case_id", "has_wsi_series", "n_wsi_series", "n_wsi_series_observed",
    "total_series_size_mb", "series_descriptions", "OS_time", "OS_event",
    "follow_up_days", "vital_status", "histology_diagnosis", "age", "sex", "race",
    "tumor_site", "tumor_size_cm", "tumor_stage_pathological",
    "pathologic_staging_primary_tumor_pt", "pathologic_staging_regional_lymph_nodes_pn",
    "pathologic_staging_distant_metastasis_pm", "residual_tumor",
]
cptac_cohort = cptac_cohort[cptac_model_cols]
cptac_cohort.to_csv(OUTPUT_DIR / "cptac_survival_cohort_candidate.csv", index=False)

print("CPTAC cohort candidate:", cptac_cohort.shape)
display(cptac_cohort.head())

## 7. 결과 해석 및 다음 단계

### 결과 해석 체크리스트
- TCGA는 `tcga_paad_matched_patient_table_dx_one_per_patient.csv`를 기준으로 환자당 diagnostic WSI 1개와 OS label을 바로 연결할 수 있다.
- CPTAC는 `follow_up_days`와 `vital_status`로 OS label을 구성할 수 있으나, 한 환자에 여러 WSI DICOM series가 있어 tumor/normal 및 대표 series 선택 기준이 필요하다.
- 모델 학습 전 patient/case 단위 split을 사용해야 하며, patch 단위 split은 데이터 누수 위험이 높다.

### 다음 단계
1. TCGA diagnostic WSI를 대상으로 baseline survival cohort를 먼저 확정한다.
2. `OS_time > 0`, `OS_event` 결측 없음, WSI 존재 조건으로 최종 inclusion/exclusion table을 만든다.
3. 병리 WSI thumbnail/patch 추출 가능성을 확인한다.
4. CoxPH 또는 clinical-only baseline을 만든 뒤, WSI feature 기반 survival model로 확장한다.

### 논문 작성 메모
Public pancreatic cancer cohorts were screened for patient-level availability of diagnostic whole-slide images and overall survival labels. Candidate cohorts were constructed without modifying raw data, and all downstream exclusions should be documented at the patient level.

In [ ]:
# CPTAC-PDAC DICOM WSI 열기/확인
# pydicom이 없으면 먼저 노트북 셀에서 다음 명령을 한 번 실행하세요.
# %pip install pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import pydicom
except ImportError as e:
    raise ImportError(
        "CPTAC-PDAC WSI는 DICOM 형식이라 pydicom이 필요합니다. "
        "노트북에서 `%pip install pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg` 실행 후 다시 실행하세요."
    ) from e

CPTAC_CASE_ID = "C3L-00017"          # 확인할 case_id를 변경하세요.
PREFERRED_SERIES_KEYWORD = "tumor"  # "tumor", "normal" 등으로 변경 가능

cptac_wsi_path = CPTAC_PATH / "matched" / "cptac_pda_matched_wsi_series.csv"
cptac_wsi_df = pd.read_csv(cptac_wsi_path)

case_series = cptac_wsi_df[cptac_wsi_df["case_id"].eq(CPTAC_CASE_ID)].copy()
if case_series.empty:
    raise ValueError(f"해당 case_id의 WSI series를 찾지 못했습니다: {CPTAC_CASE_ID}")

preferred = case_series[
    case_series["SeriesDescription"].astype(str).str.contains(PREFERRED_SERIES_KEYWORD, case=False, na=False)
]
selected_series = (preferred if not preferred.empty else case_series).sort_values("series_size_MB", ascending=True).iloc[0]

display(case_series[["case_id", "SeriesDescription", "StudyInstanceUID", "SeriesInstanceUID", "series_size_MB"]])
print("선택한 series")
display(selected_series.to_frame().T)

wsi_dicom_root = CPTAC_PATH / "WSI_DICOM"
series_dir_candidates = list(wsi_dicom_root.rglob(f"SM_{selected_series['SeriesInstanceUID']}"))
if len(series_dir_candidates) == 0:
    raise FileNotFoundError(f"DICOM series directory를 찾지 못했습니다: {selected_series['SeriesInstanceUID']}")

series_dir = series_dir_candidates[0]
dicom_files = sorted(series_dir.glob("*.dcm"), key=lambda p: p.stat().st_size)
if len(dicom_files) == 0:
    raise FileNotFoundError(f"DICOM 파일이 없습니다: {series_dir}")

print(f"series_dir: {series_dir}")
print(f"n_dicom_files: {len(dicom_files)}")

# DICOM WSI series에는 LABEL/THUMBNAIL/OVERVIEW/VOLUME instance가 함께 들어갈 수 있습니다.
# 실제 병리 조직 확인에는 LABEL이 아니라 VOLUME instance를 사용합니다.
dicom_meta = []
for dcm_path in dicom_files:
    ds_meta = pydicom.dcmread(dcm_path, stop_before_pixels=True)
    dicom_meta.append(
        {
            "file_name": dcm_path.name,
            "size_mb": dcm_path.stat().st_size / (1024 ** 2),
            "rows": getattr(ds_meta, "Rows", None),
            "columns": getattr(ds_meta, "Columns", None),
            "number_of_frames": getattr(ds_meta, "NumberOfFrames", 1),
            "image_type": " | ".join(map(str, getattr(ds_meta, "ImageType", []))).upper(),
            "photometric": getattr(ds_meta, "PhotometricInterpretation", None),
            "transfer_syntax": str(ds_meta.file_meta.TransferSyntaxUID) if hasattr(ds_meta, "file_meta") else None,
        }
    )

meta_df = pd.DataFrame(dicom_meta).sort_values("size_mb")
meta_df["is_label_or_overview"] = meta_df["image_type"].str.contains("LABEL|THUMBNAIL|OVERVIEW", na=False)
meta_df["is_volume"] = meta_df["image_type"].str.contains("VOLUME", na=False)
display(meta_df)

volume_df = meta_df[meta_df["is_volume"]].copy()
if volume_df.empty:
    raise ValueError("이 series에서 병리 tile에 해당하는 VOLUME DICOM instance를 찾지 못했습니다.")

# 빠른 육안 확인은 RESAMPLED VOLUME을 우선 사용합니다. 원본 tile 확인이 필요하면 USE_RESAMPLED_VOLUME=False로 바꾸세요.
USE_RESAMPLED_VOLUME = True
if USE_RESAMPLED_VOLUME:
    preview_candidates = volume_df[volume_df["image_type"].str.contains("RESAMPLED", na=False)]
    if preview_candidates.empty:
        preview_candidates = volume_df
else:
    preview_candidates = volume_df[~volume_df["image_type"].str.contains("RESAMPLED", na=False)]
    if preview_candidates.empty:
        preview_candidates = volume_df

preview_row = preview_candidates.sort_values(["number_of_frames", "size_mb"], ascending=[False, False]).iloc[0]
preview_path = series_dir / preview_row["file_name"]
print("선택한 병리 VOLUME instance")
display(preview_row.to_frame().T)
print(f"preview_path: {preview_path}")

ds = pydicom.dcmread(preview_path)
try:
    arr = ds.pixel_array
except Exception as e:
    raise RuntimeError(
        "DICOM pixel decoding에 실패했습니다. 압축 DICOM이면 "
        "`%pip install pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg` 설치 후 다시 실행하세요."
    ) from e

def get_frame_stack(pixel_array: np.ndarray) -> np.ndarray:
    if pixel_array.ndim == 4:
        return pixel_array
    if pixel_array.ndim == 3 and pixel_array.shape[-1] in (3, 4):
        return pixel_array[None, ...]
    if pixel_array.ndim == 3:
        return pixel_array[..., None]
    if pixel_array.ndim == 2:
        return pixel_array[None, ..., None]
    raise ValueError(f"예상하지 못한 pixel_array shape입니다: {pixel_array.shape}")


frames = get_frame_stack(arr)
n_frames = frames.shape[0]
sample_idx = np.linspace(0, n_frames - 1, min(n_frames, 64), dtype=int)
sample_frames = frames[sample_idx]

# 흰 배경/빈 tile보다 조직이 많은 tile을 보기 위해 밝기와 색 변화량으로 간단히 scoring합니다.
rgb_sample = sample_frames[..., :3].astype(np.float32)
brightness = rgb_sample.mean(axis=(1, 2, 3))
color_variation = rgb_sample.std(axis=(1, 2, 3))
tissue_score = (255 - brightness) + color_variation
top_k = min(16, len(sample_idx))
selected_idx = sample_idx[np.argsort(tissue_score)[-top_k:]][::-1]

n_cols = 4
n_rows = int(np.ceil(top_k / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows))
axes = np.array(axes).reshape(-1)
for ax, frame_idx in zip(axes, selected_idx):
    img = frames[frame_idx]
    if img.shape[-1] == 1:
        ax.imshow(img[..., 0], cmap="gray")
    else:
        ax.imshow(img[..., :3])
    ax.set_title(f"frame {frame_idx}", fontsize=9)
    ax.axis("off")
for ax in axes[len(selected_idx):]:
    ax.axis("off")
plt.suptitle(f"{CPTAC_CASE_ID} | {selected_series['SeriesDescription']} | VOLUME tile preview", y=1.02)
plt.tight_layout()
plt.show()

print("pixel_array shape:", arr.shape)
print("frame stack shape:", frames.shape)
print("selected frame indices:", selected_idx.tolist())



In [ ]:
# CPTAC-PDAC DICOM VOLUME tile을 하나의 병리 슬라이드처럼 mosaic으로 재구성
# 전제: 바로 위 셀에서 preview_path, ds, frames가 생성되어 있어야 합니다.

from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt

required_vars = ["CPTAC_CASE_ID", "selected_series", "preview_path", "ds", "frames"]
missing_vars = [name for name in required_vars if name not in globals()]
if missing_vars:
    raise RuntimeError(f"먼저 위의 DICOM VOLUME preview 셀을 실행하세요. 누락 변수: {missing_vars}")

tile_rows = int(ds.Rows)
tile_cols = int(ds.Columns)
total_rows = int(getattr(ds, "TotalPixelMatrixRows", tile_rows))
total_cols = int(getattr(ds, "TotalPixelMatrixColumns", tile_cols))
grid_rows = math.ceil(total_rows / tile_rows)
grid_cols = math.ceil(total_cols / tile_cols)
expected_frames = grid_rows * grid_cols

print("tile size:", (tile_rows, tile_cols))
print("total matrix:", (total_rows, total_cols))
print("tile grid:", (grid_rows, grid_cols))
print("frames:", frames.shape[0], "expected:", expected_frames)

if frames.shape[0] > expected_frames:
    raise ValueError("frame 수가 예상 tile grid보다 많습니다. frame 위치 metadata 확인이 필요합니다.")

# 화면 표시와 저장을 위해 너무 큰 mosaic은 tile 단위로 downsample합니다.
MAX_MOSAIC_PIXELS = 80_000_000
downsample = max(1, math.ceil(math.sqrt((total_rows * total_cols) / MAX_MOSAIC_PIXELS)))
if downsample > 1:
    print(f"mosaic이 커서 {downsample}배 downsample하여 구성합니다.")

ds_tile_rows = math.ceil(tile_rows / downsample)
ds_tile_cols = math.ceil(tile_cols / downsample)
mosaic = np.full(
    (grid_rows * ds_tile_rows, grid_cols * ds_tile_cols, frames.shape[-1]),
    fill_value=255,
    dtype=frames.dtype,
)

for frame_idx in range(frames.shape[0]):
    row_idx = frame_idx // grid_cols
    col_idx = frame_idx % grid_cols
    tile = frames[frame_idx]
    if downsample > 1:
        tile = tile[::downsample, ::downsample]
    r0 = row_idx * ds_tile_rows
    c0 = col_idx * ds_tile_cols
    mosaic[r0:r0 + tile.shape[0], c0:c0 + tile.shape[1], :tile.shape[-1]] = tile

crop_rows = math.ceil(total_rows / downsample)
crop_cols = math.ceil(total_cols / downsample)
mosaic = mosaic[:crop_rows, :crop_cols]
display_img = mosaic[..., 0] if mosaic.shape[-1] == 1 else mosaic[..., :3]

mosaic_output_path = OUTPUT_DIR / f"cptac_{CPTAC_CASE_ID}_{selected_series['SeriesDescription'].replace(' ', '_')}_mosaic.png"
plt.imsave(mosaic_output_path, display_img, cmap="gray" if display_img.ndim == 2 else None)

plt.figure(figsize=(12, 12))
plt.imshow(display_img, cmap="gray" if display_img.ndim == 2 else None)
plt.axis("off")
plt.title(f"{CPTAC_CASE_ID} | {selected_series['SeriesDescription']} | WSI mosaic")
plt.show()

print("mosaic shape:", mosaic.shape)
print("saved:", mosaic_output_path)

### 논문 작성 메모
### CPTAC-PDAC DICOM WSI는 VOLUME instance의 tile frame을 row-major grid로 재구성하여 slide-level overview를 생성하였다.


## 8. TCGA-PAAD와 CPTAC-PDAC 공통 clinical/omics 비교

### 목적
두 코호트에서 공통으로 사용할 수 있는 clinical variable을 의미 기준으로 매핑하고, TCGA RNA-seq gene symbol과 CPTAC RNA/proteome/phosphoproteome gene symbol의 교집합을 확인한다.

### 출력
- `../../results/pancreatic_cancer_pathology/data_verification/common_data/clinical_common_variable_map.csv`
- `../../results/pancreatic_cancer_pathology/data_verification/common_data/clinical_common_completeness.csv`
- `../../results/pancreatic_cancer_pathology/data_verification/common_data/tcga_common_clinical_harmonized.csv`
- `../../results/pancreatic_cancer_pathology/data_verification/common_data/cptac_common_clinical_harmonized.csv`
- `../../results/pancreatic_cancer_pathology/data_verification/common_data/omics_gene_set_summary.csv`
- `../../results/pancreatic_cancer_pathology/data_verification/common_data/common_genes_*.csv`


In [ ]:
from pathlib import Path

import pandas as pd

from scripts.compare_cptac_tcga_common_data import main as compare_common_data

common_results = compare_common_data()
COMMON_OUTPUT_DIR = OUTPUT_DIR / "common_data"

print(f"COMMON_OUTPUT_DIR: {COMMON_OUTPUT_DIR.resolve()}")
display(common_results["clinical_variable_map"])
display(common_results["gene_set_summary"])


In [ ]:
clinical_completeness = pd.read_csv(COMMON_OUTPUT_DIR / "clinical_common_completeness.csv")

clinical_completeness_pivot = clinical_completeness.pivot_table(
    index="variable",
    columns="cohort",
    values="missing_rate",
    aggfunc="first",
)
display(clinical_completeness_pivot.sort_index())

tcga_common_clinical = pd.read_csv(COMMON_OUTPUT_DIR / "tcga_common_clinical_harmonized.csv")
cptac_common_clinical = pd.read_csv(COMMON_OUTPUT_DIR / "cptac_common_clinical_harmonized.csv")

print("TCGA common clinical:", tcga_common_clinical.shape)
print("CPTAC common clinical:", cptac_common_clinical.shape)
display(tcga_common_clinical.head())
display(cptac_common_clinical.head())


In [ ]:
common_gene_files = sorted(COMMON_OUTPUT_DIR.glob("common_genes_*.csv"))
print("common gene files")
for path in common_gene_files:
    df = pd.read_csv(path)
    print(f"{path.name}: {df.shape[0]:,} genes")

common_all_omics_path = COMMON_OUTPUT_DIR / "common_genes_tcga_rna_all__cptac_rna__proteome__phosphoproteome.csv"
common_all_omics_genes = pd.read_csv(common_all_omics_path)
display(common_all_omics_genes.head(30))

### 논문 작성 메모
### TCGA-PAAD와 CPTAC-PDAC의 공통 임상 변수는 age, sex, race, vital status, OS time/event, diagnosis, pathologic stage/T/N/M로 정리하였다.
### Omics feature 비교는 gene symbol 기준으로 수행하였으며, TCGA RNA-seq와 CPTAC RNA/proteome/phosphoproteome에 모두 존재하는 gene set을 downstream multi-modal 분석 후보로 사용한다.
